<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/visualize_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!cd cursivetransformer && pip install -r requirements.txt
!wandb login

In [ ]:
import sys
sys.path.append('/content/cursivetransformer/')

import torch
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from cursivetransformer.model import get_all_args, get_checkpoint
from cursivetransformer.data import create_datasets
from cursivetransformer.sample import generate_n_words, plot_strokes, offsets_to_strokes

In [ ]:
args = get_all_args(False)
args.wandb_project = 'bigbank_2k'
args.load_from_run_id = '7e9hz1og'
args.max_seq_length = 1250
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.vocab_size = train_dataset.get_vocab_size()
args.context_vocab_size = train_dataset.get_char_vocab_size()
print(f"Dataset determined that: {args.vocab_size=}, {args.block_size=}")
model, _, _, _, _ = get_checkpoint(args, sample_only=True)

In [ ]:
x, c, y = test_dataset[0]
x = x.unsqueeze(0).to('cuda')
c = c.unsqueeze(0).to('cuda')
y = y.unsqueeze(0).to('cuda')

In [ ]:
# For CausalSelfAttention
self_attn_layer = model.transformer.h[0].attn
self_attn_patterns = {}

def self_attn_hook(mod, inp, out):
    # The attention pattern is computed but not directly returned
    # We need to recompute it here
    q, k, v = mod.c_attn(inp[0]).split(mod.n_embd, dim=2)
    B, T, C = q.size()
    k = k.view(B, T, mod.n_head, C // mod.n_head).transpose(1, 2)
    q = q.view(B, T, mod.n_head, C // mod.n_head).transpose(1, 2)
    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
    att = att.masked_fill(mod.bias[:,:,:T,:T] == 0, float('-inf'))
    att = F.softmax(att, dim=-1)
    self_attn_patterns[mod] = att.detach()

self_attn_layer.register_forward_hook(self_attn_hook)

# For CrossAttention
cross_attn_layer = model.transformer.h[0].cross_attn
cross_attn_patterns = {}

def cross_attn_hook(mod, inp, out):
    # The attention pattern is computed but not directly returned
    # We need to recompute it here
    x, context = inp
    B, T, C = x.size()
    _, T_ctx, _ = context.size()
    q = mod.c_attn_q(x).view(B, T, mod.n_ctx_head, C // mod.n_ctx_head).transpose(1, 2)
    k, v = mod.c_attn_kv(context).split(mod.n_embd_context, dim=2)
    k = k.view(B, T_ctx, mod.n_ctx_head, C // mod.n_ctx_head).transpose(1, 2)
    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
    att = F.softmax(att, dim=-1)
    cross_attn_patterns[mod] = att.detach()

cross_attn_layer.register_forward_hook(cross_attn_hook)

# Now run your model
with torch.no_grad():
    output = model(x, c)

# Access the attention patterns
print("Self-attention pattern shape:", next(iter(self_attn_patterns.values())).shape)
print("Cross-attention pattern shape:", next(iter(cross_attn_patterns.values())).shape)

In [ ]:
self_attn_patterns.values()

In [ ]:
  w